In [ ]:
import requests
import json
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from io import BytesIO
from openai import OpenAI
import os
from dotenv import load_dotenv

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# --- 1. Setup API and Prompt ---
load_dotenv()
OPENROUTER_API_KEY = os.getenv("My_Openrouter_key")
image_url = "https://www.shutterstock.com/image-vector/set-realistic-food-grocery-store-260nw-2681430175.jpg"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

prompt_content = (
    "Analyze this image and identify all distinct objects. Respond ONLY with a valid JSON array of objects. "
    "Each object must have 'name', 'category', and a 'box' field in [xmin, ymin, xmax, ymax] format (scaled 0-1000)."
)

# --- 2. Request Object Detection ---
try:
    completion = client.chat.completions.create(
        model="nvidia/nemotron-nano-12b-v2-vl:free",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_content},
                {"type": "image_url", "image_url": {"url": image_url}}
            ]
        }]
    )

    if not completion or not completion.choices:
        raise ValueError("API returned an empty response.")

    model_output = completion.choices[0].message.content

    # --- 3. Clean and Parse ---
    # Remove markdown code block markers if present
    clean_output = model_output.strip()
    if clean_output.startswith("```json"):
        clean_output = clean_output.split("```json")[1].split("```")[0].strip()
    elif clean_output.startswith("```"):
        clean_output = clean_output.split("```")[1].split("```")[0].strip()

    detections = json.loads(clean_output)

    # --- 4. Visualize ---
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content))
    draw = ImageDraw.Draw(img)
    w, h = img.size

    print(f"Detected {len(detections)} object(s).")

    for det in detections:
        xmin, ymin, xmax, ymax = det['box']
        left, top = (xmin * w / 1000), (ymin * h / 1000)
        right, bottom = (xmax * w / 1000), (ymax * h / 1000)

        draw.rectangle([left, top, right, bottom], outline="lime", width=3)
        draw.text((left + 5, top + 5), f"{det['name']}", fill="lime")

    plt.figure(figsize=(12, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Cleaned JSON Object Detection")
    plt.show()

except json.JSONDecodeError:
    print("Error parsing JSON. Raw output was:", model_output)
except Exception as e:
    print(f"An error occurred: {e}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/nothing/pytorch/default/1/best (1).pt'